# Trajectory chatter diagnostics — single-trajectory deep dive (profiles era)

Visual companion to `chatter_diagnostics.py`. The script computes the metrics over the whole sweep; this notebook lets you **look at one trajectory** and see *why* it scored the way it did. Every band and metric here calls the same kernels in `chatter_diagnostics.py` — nothing is re-implemented, so what you see matches what the batch screen decides.

## The question this answers

The ground-truth wheel forces look noisy by eye, but that ripple has **four** origins — two are signal the PINN must learn, two are unwanted:

| Source | Frequency set by | In `Msw`? | θ-locked? | Verdict |
|---|---|---|---|---|
| Roller-switching ripple | `|ω|` (sweeps) | no | **yes** | keep |
| LuGre stick–slip | `Vp` (slip) | no | no | **keep** |
| ASMC chatter | controller `eps` (≈fixed) | **yes** | no | flag (kept+flagged) |
| Numerical hash | broadband | no | no | flag (reject) |

Each metric below isolates one of these axes. See `Trajectory_Chatter_Diagnostics_PLAN.md` for the full derivation.

In [ ]:
import importlib
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyarrow.feather as feather
from scipy.signal import stft, welch, coherence, butter, sosfiltfilt, hilbert

import chatter_diagnostics as cd
importlib.reload(cd)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.titlesize': 11, 'legend.fontsize': 8})
cfg = cd.ChatterConfig()
print('config:', cfg)

## 1. Pick a trajectory

Point `DATA_DIR` at the sweep output. Set `TARGET_FILE` to a specific filename, or leave it `None` to take the first Arrow file. **If no real data exists yet** (the production sweep is gated on the solver-ablation winner), the notebook falls back to a synthetic trajectory of type `DEMO_KIND` so every cell still runs — switch `DEMO_KIND` between `clean / chatter / hash / lugre_heavy / lugre_above` to see how the plots and metrics respond.

In [ ]:
# === EDIT THESE ===
DATA_DIR    = Path(r'..\data\Simulation_Data_MecanumSlipSpin_LugreAdamov')
TARGET_FILE = None              # None -> first arrow in DATA_DIR; else a filename
WHEEL       = 0                 # 0..3 — which wheel to deep-dive
DEMO_KIND   = 'lugre_above'     # synthetic fallback: clean|chatter|hash|lugre_heavy|lugre_above
# ===================

_NAME_BY_KIND = {'clean': 'octagon', 'chatter': 'spin_creep', 'hash': 'coupled_vomega',
                 'lugre_heavy': 'spiral_orbit', 'lugre_above': 'long_circle'}

paths = sorted(DATA_DIR.glob('*.arrow')) if DATA_DIR.exists() else []
paths = [p for p in paths if cd.parse_arrow_filename(p.name)]
if TARGET_FILE:
    path = DATA_DIR / TARGET_FILE
elif paths:
    path = paths[0]
else:
    import test_chatter_diagnostics as tt
    _tmp = Path(tempfile.mkdtemp())
    fn = _NAME_BY_KIND[DEMO_KIND] + '_c001_mu_0.4_case1_lugre_adamov_chi_0.005.arrow'
    path = _tmp / fn
    feather.write_feather(tt.make_traj(DEMO_KIND), path)
    print('[demo] no real data found -> synthetic', repr(DEMO_KIND))

df   = feather.read_feather(path)
meta = cd.parse_arrow_filename(path.name)
print('file:', path.name)
print('meta:', meta)

In [ ]:
# Per-wheel arrays + the chosen wheel's physical bands (same as the kernels see)
t  = df['time'].to_numpy(float)
fs = 1.0 / float(np.median(np.diff(t)))
W  = {s: df[[f'{s}_{k}' for k in range(1, 5)]].to_numpy(float)
      for s in ('Fpar', 'Fperp', 'util', 'Msw', 'Meq', 'Vpx', 'Vpy', 'wz', 'Mz', 'Msat')}
omega = df[[f'w{k}' for k in range(1, 5)]].to_numpy(float)
theta = df[[f'theta{k}' for k in range(1, 5)]].to_numpy(float)
vp    = np.hypot(W['Vpx'], W['Vpy'])

i  = WHEEL
mu = meta['mu']
is_coulomb = abs(meta['chi']) < cfg.chi_coulomb_tol
oa, va = np.abs(omega[:, i]), vp[:, i]
f_hash = cd.hash_cutoff_hz(oa, va, mu, cfg)
print(f'fs = {fs:.0f} Hz   N = {len(t)}   wheel = {i + 1}   chi = {meta["chi"]}   '
      f'coulomb-branch = {is_coulomb}')
print(f'F_hash (per-trajectory hash ceiling) = {f_hash:.1f} Hz')

## 2. Time-domain overview

Before any spectral work, glance at the raw signals — this catches solver blow-ups, dropouts, or unit issues that would corrupt every metric downstream. Watch for: `Fpar/Fperp` riding a clean slow envelope with fine ripple on top (good), versus fuzzy high-frequency fur with no structure (suspect). `util` near 1.0 marks friction-circle saturation, where both stick-slip and ASMC chatter get violent. `Msw` (the switching wrench) should be small and smooth; a buzzing `Msw` is the first sign of control chatter.

In [ ]:
fig, ax = plt.subplots(5, 1, figsize=(12, 10), sharex=True)
ax[0].plot(t, W['Fpar'][:, i], lw=0.6, label='Fpar'); ax[0].plot(t, W['Fperp'][:, i], lw=0.6, label='Fperp')
ax[0].set_ylabel('force (N)'); ax[0].legend(ncol=2); ax[0].set_title(f'{path.name}  — wheel {i + 1}')
ax[1].plot(t, W['Msw'][:, i], lw=0.6, label='Msw (switching)'); ax[1].plot(t, W['Meq'][:, i], lw=0.8, label='Meq (equivalent)')
ax[1].set_ylabel('torque (N·m)'); ax[1].legend(ncol=2)
ax[2].plot(t, W['util'][:, i], lw=0.7, color='C3'); ax[2].axhline(0.8, color='gray', ls=':'); ax[2].axhline(1.0, color='r', ls='--')
ax[2].set_ylabel('util = |F|/μN')
ax[3].plot(t, va, lw=0.7, color='C2'); ax[3].set_ylabel('|Vp| (m/s)')
ax[4].plot(t, oa, lw=0.7, color='C4'); ax[4].set_ylabel('|ω| (rad/s)'); ax[4].set_xlabel('time (s)')
plt.tight_layout(); plt.show()

## 3. The physical frequency bands — `f_roll`, `f_lugre`, `F_hash`

Everything downstream rests on a per-sample estimate of *how high in frequency the physics can legitimately reach*. Two physical processes set it:

- **Roller ripple** lives at `f_roll(t) = 12·|ω|/2π = 1.910·|ω|` Hz and its harmonics. The 3rd harmonic `3·f_roll` is the top of the roller band.
- **LuGre stick–slip** is a slip-driven low-pass with corner `f_lugre ≈ σ₀·Vp/(g_t·2π) ≈ 653·Vp` Hz (σ₀ = 1640 1/m, g_t ≈ μ). The force tracks slip below this corner.

The **LuGre-aware ceiling** is `f_phys(t) = max(3·f_roll, 1.5·f_lugre, 40 Hz)`, and the per-trajectory **hash cutoff** is its 95th percentile, `F_hash`. The crucial point (which diag-1's fixed `3·f_roll` rule got wrong): `f_lugre` can climb *above* the 3rd roller harmonic at high slip — 196 Hz at Vp=0.3 vs 86 Hz at |ω|=15 — so the ceiling has to be computed from the slip state, not assumed. Energy **above** `F_hash` is what the hash metric (M2) interrogates.

In [ ]:
fr = cd.f_roll_series(oa)
fl = cd.f_lugre_series(va, mu)
f_phys = np.maximum.reduce([cfg.ridge_n_harmonics * fr, cfg.k_lugre * fl,
                            np.full_like(fr, cfg.f_floor_hz)])
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(t, fr, lw=0.8, label='f_roll = 1.910·|ω|')
ax.plot(t, cfg.ridge_n_harmonics * fr, lw=0.8, label='3·f_roll (top roller harmonic)')
ax.plot(t, fl, lw=0.8, label='f_lugre ≈ 653·Vp')
ax.plot(t, f_phys, 'k', lw=1.6, label='f_phys = max(3·f_roll, 1.5·f_lugre, 40)')
ax.axhline(f_hash, color='r', ls='--', lw=1.2, label=f'F_hash (p95 of f_phys) = {f_hash:.0f} Hz')
ax.set_xlabel('time (s)'); ax.set_ylabel('frequency (Hz)')
ax.set_title('Physical frequency bands and the hash ceiling'); ax.legend(fontsize=8); plt.show()

## 4. M1 — `ridge_concentration`  (roller ripple = SIGNAL, higher is better)

**What it measures:** the fraction of off-DC STFT energy of `Fpar` that lies within ±25% of a roller harmonic `h·f_roll(t)`, h = 1..3. Computed on the ×8-decimated 250 Hz grid (Nyquist 125 Hz covers the 3rd harmonic up to |ω| ≈ 22 rad/s).

**Physics:** the roller handoff is `sawtooth_tanh(θ)` with period 2π/12, so the force ripples at exactly `f_roll`, and the ridge **sweeps with wheel speed** — the unmistakable fingerprint of physical ripple.

**How to read the spectrogram:** bright energy hugging the cyan ridge (and the dashed harmonics) = clean roller ripple. A **flat horizontal band** that ignores ω = ASMC chatter at a fixed frequency. **Broadband haze above the ridge** = numerical hash.

**Caveat:** near stall (`|ω| → 0`) the ±floor widens the band relative to the signal, so even white noise scores ~0.3 — this is why the classifier never passes on ridge concentration alone.

In [ ]:
fac = cfg.decimate_factor
fs_d = fs / fac
fpar_d = cd._decimate_cols(W['Fpar'], fac)[:, i]
om_d   = np.abs(cd._decimate_cols(omega, fac)[:, i])
t_d    = t[::fac][:len(fpar_d)]
fc = cd._hp_cutoff(om_d, fs_d, cfg)
sig = cd._highpass(fpar_d, fs_d, fc)
nper = min(cfg.stft_nperseg, len(sig))
f, tt, Z = stft(sig, fs=fs_d, nperseg=nper, noverlap=nper - max(1, nper // 8),
                detrend='constant', boundary=None, padded=False)
mag_db = 20 * np.log10(np.abs(Z) + 1e-12)
tt_abs = tt + t_d[0]
fr_stft = np.interp(tt_abs, t_d, cd.f_roll_series(om_d))

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.pcolormesh(tt_abs, f, mag_db, shading='auto', cmap='magma',
                   vmin=mag_db.max() - 50, vmax=mag_db.max())
for h in range(1, cfg.ridge_n_harmonics + 1):
    ax.plot(tt_abs, h * fr_stft, color='cyan' if h == 1 else 'white', lw=1.1,
            ls='-' if h == 1 else '--', label=f'{h}·f_roll')
ax.set_ylim(0, fs_d / 2); ax.set_xlabel('time (s)'); ax.set_ylabel('freq (Hz)')
plt.colorbar(im, ax=ax, label='dB'); ax.legend(loc='upper right')
m1 = cd.m1_ridge_concentration(fpar_d, om_d, fs_d, cfg)
ax.set_title(f'STFT of Fpar (HP @ {fc:.1f} Hz) with roller ridge   —   M1 ridge_concentration = {m1:.3f}')
plt.show()

## 5. M2 — `hash_fraction`  (the core chatter detector, lower is better)

**What it measures:** the fraction of off-DC PSD energy of `Fpar` that sits **above `F_hash`** — i.e. above what roller ripple and LuGre can physically produce. Computed on the **full 2000 Hz** signal (decimating first would low-pass away the very chatter we are screening for — a latent flaw in diag-1, which worked at 200 Hz).

**Interpretation:** energy above the physical ceiling is, by construction, unexplained by either physical oscillation. A high `hash_fraction` is the primary symptom of unwanted chatter — but it does **not** by itself decide *which* kind: that takes M4 (coherent with the controller?) and M5 (slip-modulated LuGre that poked above the cutoff?). The shaded region is the band counted as hash.

In [ ]:
f, P = cd._welch_psd(W['Fpar'][:, i], fs, cfg.welch_nperseg)
m2 = cd.m2_hash_fraction(W['Fpar'][:, i], fs, f_hash, cfg)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.semilogy(f, P + 1e-20, lw=0.7)
ax.axvline(f_hash, color='r', ls='--', label=f'F_hash = {f_hash:.0f} Hz')
ax.axvspan(f_hash, fs / 2, color='r', alpha=0.10, label='hash band (counted by M2)')
ax.set_xlabel('freq (Hz)'); ax.set_ylabel('PSD of Fpar'); ax.legend()
ax.set_title(f'Welch PSD of Fpar @ {fs:.0f} Hz   —   M2 hash_fraction = {m2:.3f}')
plt.show()

## 6. M3 — `control_chatter_index`  (ASMC switching fingerprint, lower is better)

**What it measures:** `sqrt(HF power of Msw above F_hash) / (RMS(Meq) + ε)`. The switching wrench `Msw = −K·tanh(s/eps)` is where ASMC chatter is *born*; the equivalent control `Meq` is smooth. A high ratio means the controller itself is limit-cycling.

**Why this is the cleanest separator the new columns unlock:** roller ripple and LuGre are *friction outputs* — the controller never commands them, so they contribute **zero** to `Msw` HF energy. Only genuine control chatter shows up here. Note `eps = max(0.025·PEAK_V, 0.001)`, so low-speed profiles (e.g. `spin_creep`) get the thinnest boundary layer and are the most chatter-prone — expect M3 to run higher there.

The plot overlays the `Msw` and `Fpar` PSDs: if both light up *together* in the band above `F_hash`, the chatter is propagating from control into the physics.

In [ ]:
f, Pmsw = cd._welch_psd(W['Msw'][:, i], fs, cfg.welch_nperseg)
_, Pf   = cd._welch_psd(W['Fpar'][:, i], fs, cfg.welch_nperseg)
m3 = cd.m3_control_chatter_index(W['Msw'][:, i], W['Meq'][:, i], fs, f_hash, cfg)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.semilogy(f, Pmsw + 1e-20, lw=0.7, label='PSD Msw (switching)')
ax.semilogy(f, Pf + 1e-20, lw=0.7, alpha=0.7, label='PSD Fpar (force)')
ax.axvline(f_hash, color='r', ls='--', label=f'F_hash = {f_hash:.0f} Hz')
ax.axvspan(f_hash, fs / 2, color='r', alpha=0.08)
ax.set_xlabel('freq (Hz)'); ax.set_ylabel('PSD'); ax.legend()
ax.set_title(f'Msw vs Fpar spectra   —   M3 control_chatter_index = {m3:.3f}')
plt.show()

## 7. M4 — `msw_force_coherence_hf`  (splits coherent chatter from incoherent hash)

**What it measures:** the `Msw`-power-**weighted** magnitude-squared coherence γ²(Msw, Fpar) over the band above `F_hash`. Weighting by the `Msw` PSD asks the right question — *where the switching wrench has HF energy, is the force coherent with it?* — so a narrow coherent chatter tone gets full weight instead of being averaged away against the quiet rest of the band (the bug the synthetic test caught).

**The decision it drives:**
- **high coherence + high M2/M3** → the HF force energy is a faithful response to switching → **coherent ASMC chatter** (kept, but flagged).
- **low coherence + high M2** → HF energy not explained by any input → **numerical hash** (hard reject).

The plot shows γ²(f) with the `Msw` PSD (the weighting) on a twin axis; read the coherence *where the orange weighting is tall*.

In [ ]:
nper = min(cfg.coh_nperseg, len(t))
fco, Cxy = coherence(W['Msw'][:, i], W['Fpar'][:, i], fs=fs, nperseg=nper)
_, Pw = welch(W['Msw'][:, i], fs=fs, nperseg=nper, detrend='constant')
m4 = cd.m4_msw_force_coherence_hf(W['Msw'][:, i], W['Fpar'][:, i], fs, f_hash, cfg)
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(fco, Cxy, lw=0.8, label='coherence γ²(Msw, Fpar)')
ax.axvline(f_hash, color='r', ls='--'); ax.axvspan(f_hash, fs / 2, color='r', alpha=0.06)
ax.set_xlabel('freq (Hz)'); ax.set_ylabel('coherence'); ax.set_ylim(0, 1.02)
ax2 = ax.twinx(); ax2.semilogy(fco, Pw + 1e-20, color='C1', lw=0.6, alpha=0.6)
ax2.set_ylabel('PSD Msw (weighting)', color='C1')
ax.set_title(f'HF coherence (Msw-weighted over f > F_hash)   —   M4 = {m4:.3f}')
ax.legend(loc='upper left'); plt.show()

## 8. M5 — `hf_slip_modulation`  (the LuGre rescue)

**What it measures:** the Pearson correlation between the **envelope** of `Fpar` band-passed in `[F_hash, 0.45·fs]` (the same band M2 flags) and the slip magnitude `|Vp|`.

**Why it exists:** high-slip stick–slip can legitimately push LuGre energy *above* the percentile ceiling. That energy is genuine signal, not hash — and the tell is that its amplitude is **modulated by slip**: when `|Vp|` rises, the LuGre force swells. So:
- **high correlation** → the above-ceiling energy is slip-modulated LuGre → **veto the hash verdict** (keep).
- **flat / zero correlation** → white energy with no slip envelope → confirm hash.

This is the guard that keeps genuine stick–slip out of the reject bin. The top panel overlays the HF envelope and `|Vp|` (twin axis); the bottom is their scatter — a tilted cloud means modulation.

In [ ]:
bp_lo, bp_hi = f_hash, 0.45 * fs
if bp_hi > bp_lo:                       # guard: high-slip traj can push F_hash above 0.45*Nyquist
    sos = butter(4, [bp_lo, bp_hi], btype='bandpass', fs=fs, output='sos')
    env = np.abs(hilbert(sosfiltfilt(sos, W['Fpar'][:, i])))
    note = ''
else:
    env = np.zeros_like(t)              # no above-ceiling band exists (F_hash >= 0.9*Nyquist)
    note = '   (F_hash >= 0.9*Nyquist: no above-ceiling band — M5 N/A)'
m5 = cd.m5_hf_slip_modulation(W['Fpar'][:, i], va, oa, fs, f_hash, cfg)
fig, ax = plt.subplots(2, 1, figsize=(11, 6))
ax[0].plot(t, env, lw=0.6, color='C0', label='HF-force envelope (f > F_hash)')
axb = ax[0].twinx(); axb.plot(t, va, lw=0.8, color='C2', alpha=0.8, label='|Vp|')
ax[0].set_xlabel('time (s)'); ax[0].set_ylabel('HF env', color='C0'); axb.set_ylabel('|Vp| (m/s)', color='C2')
ax[0].set_title(f'HF envelope vs slip   —   M5 hf_slip_modulation = {m5:+.3f}{note}')
ax[1].scatter(va, env, s=3, alpha=0.3); ax[1].set_xlabel('|Vp| (m/s)'); ax[1].set_ylabel('HF env')
plt.tight_layout(); plt.show()

## 9. M6 — `theta_fold_tightness`  (ripple periodicity in θ, lower is better)

**What it measures:** fold the relevant force channel over `θ mod 2π/12` (the roller pitch, ±15°) and report `median-bin IQR / range(bin medians)`. Physical roller ripple comes from `sawtooth_tanh(θ)`, so it is **strictly periodic in θ** and collapses onto a tight closed curve when folded; anything that does not fold cleanly is non-physical and smears the band.

**χ branch (matches the kernel):** for χ > 0 the fold uses `Mz` over the full pitch (it has a clean odd-symmetric phase profile); for χ = 0 (pure Coulomb) `Mz ≡ 0`, so it falls back to high-passed `Fperp` at the pitch edges where the roller perturbation lives.

This is *independent* evidence to the STFT: M1 looks at time–frequency structure, M6 at kinematic-phase structure. On a clean trajectory they agree.

In [ ]:
half = (2 * np.pi / cd.N_ROLLERS) / 2.0
fperp_d = cd._decimate_cols(W['Fperp'], fac)[:, i]
mz_d    = cd._decimate_cols(W['Mz'], fac)[:, i]
theta_d = theta[::fac, i][:len(fperp_d)]   # chosen wheel only (single-wheel kernel)
if is_coulomb:
    fold_sig = cd._highpass(fperp_d, fs_d, cd._hp_cutoff(om_d, fs_d, cfg)); ch = 'HP Fperp (χ=0)'
else:
    fold_sig = mz_d; ch = 'Mz (χ>0)'
th_mod = np.mod(theta_d + half, 2 * half) - half
m6 = cd.m6_theta_fold_tightness(fold_sig, theta_d, fs_d, om_d, is_coulomb, cfg)
fig, ax = plt.subplots(figsize=(7, 4))
sc = ax.scatter(np.degrees(th_mod), fold_sig, c=t_d, cmap='viridis', s=3, alpha=0.35)
ax.axvline(0, color='gray', ls=':'); plt.colorbar(sc, ax=ax, label='time (s)')
ax.set_xlabel('θ mod (2π/12)  — roller-centered (deg)'); ax.set_ylabel(ch)
ax.set_title(f'θ-fold of {ch}   —   M6 theta_fold_tightness = {m6:.3f}')
plt.show()

## 10. M0 — `util_sat_frac`  (near-free saturation pre-screen)

**What it measures:** the fraction of time the per-wheel friction-circle utilization `util = |F|/(μN)` exceeds 0.8 (the master-gate budget the profiles were designed against) and 1.0 (a budget violation), max over wheels. Uses the saved `util_i` directly — no spectral work.

**Role:** sustained `util ≈ 1` is exactly where stick-slip, ASMC chatter, and solver thrashing all peak. It is not a reject on its own (some profiles legitimately ride the budget), but it contextualizes a high hash score and gates a `marginal` label.

In [ ]:
util_max = W['util'].max(axis=1)
frac_08 = float(np.mean(util_max > 0.8)); frac_10 = float(np.mean(util_max > 1.0))
fig, ax = plt.subplots(figsize=(11, 3))
for k in range(4):
    ax.plot(t, W['util'][:, k], lw=0.5, alpha=0.7, label=f'util_{k + 1}')
ax.axhline(0.8, color='gray', ls=':'); ax.axhline(1.0, color='r', ls='--')
ax.set_xlabel('time (s)'); ax.set_ylabel('util'); ax.legend(ncol=4, fontsize=7)
ax.set_title(f'Friction-circle utilization   —   frac>0.8 = {frac_08:.2f}   frac>1.0 = {frac_10:.2f}')
plt.show()

## 11. M7 — `msat_burstiness`  (localized control-torque burst)

**Why this metric exists — the spectral gates' blind spot.** When a trajectory loses tracking because it exceeds the friction circle, the saturated controller can erupt in a brief, violent oscillation of the **control torque** `Msat` (an ASMC limit cycle), usually at the very end of the run. This is real chatter, but M1–M6 **miss it** for two compounding reasons:

1. **It's localized.** A ~1 s burst in a ~22 s trajectory contributes almost nothing to the whole-trajectory PSD/coherence those metrics average over — it washes out.
2. **`F_hash` goes blind.** A friction-circle violation means gross slip (`|Vp|` up to several m/s), and `F_hash ∝ f_lugre ∝ |Vp|` inflates *above the Nyquist*, so M2/M3 examine an empty band and return exactly 0. (Verified on coupled_vomega c125: `F_hash` ≈ 3088 Hz ≫ 1000 Hz Nyquist, so `hash_fraction` = `control_chatter_index` = 0 even though `Msat` bursts at t ≈ 21 s.)

**How M7 sidesteps both.** It works on the **control torque**, not the force — the controller never commands roller ripple or LuGre, so any HF in `Msat` is genuinely control oscillation, and there is no `F_hash` involved. And it measures **localization, not amplitude**:

- **High-pass at `burst_hp_hz` = 15 Hz** ≈ 5× the wheel mechanical corner `p1/(2π·Jw) = 0.11/(2π·5.87e-3) ≈ 3 Hz`. Above the wheel corner the heavy inertia low-passes the torque, so HF content there does no useful tracking work; 15 Hz sits safely above all *functional* control (commanded references are ≤ 1 Hz) and below the measured ~29 Hz burst (and the 77 Hz roller-dynamics corner). The band `[15 Hz, Nyquist]` therefore isolates exactly the non-functional HF where a give-up burst lives.
- **Sliding-window RMS** (`burst_win_sec` = 0.25 s) of the high-passed torque, then
- **`burstiness = max_window / (p10_window + floor)`** — the worst window relative to the **quiet baseline**, taken as the 10th-percentile window (`burst_baseline_pctile` = 10), with `floor` = `BURST_FLOOR` = 0.10 N·m.

**Why the p10 baseline, not the median.** The denominator must estimate the *quiet* baseline. When the disturbed fraction (startup chatter + an end-of-run burst) exceeds ~50 % of the run, the **median** is dragged up *into* the disturbed population and the ratio collapses — strong coupled_vomega bursts scored ~93 under the old `median + 1e-3` denominator and slipped under the then-`tau = 100` gate. The 10th-percentile window-RMS is robust to a large disturbed fraction; `floor` (= 1 % of `Max_torque` = 10 N·m) is a physical silent-torque level, so a truly-silent baseline (`p10 → 0`) doesn't turn the ratio into `max/eps` and explode on a clean blip.

**Why the ratio, not the amplitude.** Broadband-reference profiles (multisine) drive *legitimately* oscillatory control all the time, so their absolute HF amplitude is high (~0.1 of the torque cap — comparable to a real burst). But that HF is **flat** (pervasive), so `max ≈ p10` → burstiness ≈ 1. A genuine burst is a **transient spike** on a quiet baseline → `max ≫ p10` → burstiness well above the gate:

| | well-tracked multisine | clean octagon | c125 (real burst) |
|---|---|---|---|
| `msat_burstiness` | 1.25 | 2.33 | **46.6** |

(Absolute HF amplitude FAILS here — multisine ≈ a real burst at ~0.1 N·m — but the ratio separates them by more than an order of magnitude. Under the new p10/`floor`=0.10 denominator the burst scale is *tens*, not the *thousands* the old `median`+1e-3 produced.)

**Reading the plot.** Top: `Msat(t)` with the ±`Max_torque` rails. Bottom: the per-window HF-RMS with the p10 baseline (dashed) and the max window (▼). A flat bottom trace ⟹ no burst; a single tall spike above the baseline ⟹ a localized burst.

**Role.** M7 is reported as an **independent `burst_flag`** (threshold `tau_burst` = 10: `burstiness > 10`), *not* folded into the spectral verdict — a trajectory can be `clean`/`chatter` in the forces yet carry a control burst. It enters the training decision only **in combination with tracking**: the blend rejects (`reject_burst`) **only when a burst co-occurs with a `missed`/`marginal` tracking verdict** — a give-up burst confirmed by lost tracking. A burst on a *well-tracked* run is kept (labelled `burst`); this is what preserves spin_creep's 66 by-design high-yaw-rate pulse-edge bursts (a tight 17–19 cluster, all tracked), versus the coupled_vomega/octagon give-up bursts (scattered 13–50, all missed).

In [ ]:
fcb = min(cfg.burst_hp_hz, 0.45 * fs)
sos_b = butter(4, fcb, btype='highpass', fs=fs, output='sos')
hf_b = np.abs(sosfiltfilt(sos_b, W['Msat'][:, i]))
winb = max(1, int(cfg.burst_win_sec * fs)); nb = len(hf_b) // winb
wr = np.sqrt((hf_b[:nb * winb].reshape(nb, winb) ** 2).mean(axis=1))
tw = t[:nb * winb:winb][:len(wr)]
burstiness, abs_norm = cd.m7_msat_burstiness(W['Msat'][:, i], fs, cfg)

fig, ax = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
ax[0].plot(t, W['Msat'][:, i], lw=0.5)
ax[0].axhline(cd.MAX_TORQUE, color='gray', ls=':'); ax[0].axhline(-cd.MAX_TORQUE, color='gray', ls=':')
ax[0].set_ylabel('Msat (N·m)'); ax[0].set_title(f'Control torque Msat — wheel {i + 1}')
ax[1].plot(tw, wr, lw=1.0, color='C3', label=f'HF-RMS (>{fcb:.0f} Hz) per {cfg.burst_win_sec}s window')
ax[1].axhline(np.median(wr), color='gray', ls='--', label='median window')
ax[1].plot(tw[np.argmax(wr)], wr.max(), 'kv', ms=9, label='max window (the burst)')
ax[1].set_xlabel('time (s)'); ax[1].set_ylabel('windowed HF-RMS'); ax[1].legend(fontsize=8)
flag = burstiness > cd.CLASSIFIER_THRESHOLDS['tau_burst']
ax[1].set_title(f'M7  burstiness = max/median = {burstiness:.1f}   (abs = {abs_norm:.3f})   burst_flag = {flag}')
plt.tight_layout(); plt.show()

## 12. Verdict for this trajectory

Finally, run the exact batch kernel (`cd.diagnose_file`) on this file and print the full metric row, the verdict, and the classifier thresholds. The verdict here is identical to what the sweep would assign. Recall the policy: **`hash` is the only hard reject; `chatter` is kept-with-flag; `clean`/`marginal` are kept.** `burst_flag` (M7) is reported independently. Thresholds are first-principles placeholders until the sweep+histogram pass sets per-profile cutoffs.

In [ ]:
row = cd.diagnose_file(path, cfg)
print('--- metrics (median over wheels; M7 max over wheels) ---')
for k in ['ridge_concentration', 'hash_fraction', 'control_chatter_index',
          'msw_force_coherence_hf', 'hf_slip_modulation', 'theta_fold_tightness',
          'util_sat_frac_0p8', 'util_sat_frac_1p0', 'msat_burstiness', 'msat_burst_abs',
          'f_hash_med']:
    v = row.get(k)
    print(f'  {k:<26} {v:.4f}' if isinstance(v, float) else f'  {k:<26} {v}')
print(f"\n  VERDICT = {row['verdict']!r}   chatter_flag = {row['chatter_flag']}   "
      f"burst_flag = {row['burst_flag']}")
print('\n--- classifier thresholds (PLACEHOLDERS — retune from the sweep) ---')
for k, v in cd.CLASSIFIER_THRESHOLDS.items():
    print(f'  {k:<18} {v}')